In [1]:
!pip install -q -U selenium beautifulsoup4 requests pandas tqdm
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y -qq ./google-chrome-stable_current_amd64.deb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 8.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which i

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import pandas as pd
import requests
import logging
import time
import re

In [3]:
logging.basicConfig(filename='GP_logs.log', level=logging.INFO,format='%(asctime)s %(message)s',datefmt='%Y-%m-%d %H:%M:%S', force=True)
logging.info('начат парсинг хабра')

In [5]:
category_list = ['AI', 'CRM-системы', 'Data Science', 'DevOps', 'SRE', 'Web-верстка',
                 'Администрирование баз данных', 'Аналитика', 'Внедрение и сопровождение ПО',
                 'Игровое ПО / Геймдев', 'Инжиниринг', 'Интернет, создание и поддержка сайтов',
                 'Информационная безопасность', 'Киберспорт', 'Компьютерная анимация и мультимедиа',
                 'Контент', 'Мобильная разработка', 'Нейросети / Искусственный интеллект',
                 'Оптимизация, SEO', 'Передача данных и доступ в интернет', 'Разработка и сопровождение банковского ПО',
                 'Разработка, программирование', 'Сетевые технологии', 'Системная интеграция', 'Системное администрирование',
                 'Системы автоматизированного проектирования (САПР)', 'Системы управления предприятием (ERP)', 'Сотовые, беспроводные технологии',
                 'Телекоммуникации и связь', 'Тестирование, QA', 'Техническая документация', 'Техническая поддержка', 'Управление продуктом',
                 'Управление проектами', 'Управление разработкой', 'Юзабилити', 'Другое', 'Начало карьеры, мало опыта']
target = 11000
links = 15000
max_pg = 40
slp1 = 2
slp2 = 0.2

In [6]:
opt = Options()
opt.add_argument('--headless=new')
opt.add_argument('--no-sandbox')
opt.add_argument('--disable-dev-shm-usage')
opt.add_argument('--window-size=1920,1080')
opt.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)')
br = webdriver.Chrome(options=opt)
logging.info('хром драйвер запущен')

In [8]:
d = []
for i in tqdm(category_list):
    if len(d) >= links:
        break
    for p in range(1, max_pg + 1):
        u = f'https://career.habr.com/vacancies?q={i.replace(" ", "+")}&page={p}'
        logging.info(f'отправлен запрос к вакансиям хабр на странице {p}')
        try:
            br.get(u)
        except Exception as err:
            logging.error(f'ошибка загрузки браузера {err}')
            break
        time.sleep(slp1)
        sp = BeautifulSoup(br.page_source, 'html.parser')
        cs = sp.find_all('div', class_='vacancy-card')
        if not cs:
            break
        for cd in cs:
            a = cd.find('a', class_='vacancy-card__title-link')
            if a is None:
                continue
            h = a.get('href', '')
            if not h:
                continue
            if h.startswith('/'):
                ln = 'https://career.habr.com' + h
            else:
                ln = h
            t = cd.find(class_='vacancy-card__title')
            f = cd.find(class_='vacancy-card__company')
            s = cd.find(class_='vacancy-card__salary')
            prof = t.get_text(' ', strip=True) if t else ''
            firm = f.get_text(' ', strip=True) if f else ''
            sal = s.get_text(' ', strip=True) if s else ''
            m = []
            for z in cd.select('.vacancy-card__meta .chip-with-icon__text'):
                x = z.get_text(' ', strip=True)
                if x:
                    m.append(x)
            for z in cd.select('.vacancy-card__meta .chip-without-icon__text'):
                x = z.get_text(' ', strip=True)
                if x:
                    m.append(x)
            d.append({
                'vacancy_category': i,
                'profession': prof,
                'firm_name': firm,
                'salary_text': sal,
                'meta_tokens': m,
                'link': ln
            })
            if len(d) >= links:
                break
        logging.info(f'успешно для категории {i}, на страницу {p}, карточки {len(cs)}')
        if len(d) >= links:
            break
logging.info(f'ссылок собрано {len(d)}')

  0%|          | 0/38 [00:00<?, ?it/s]

In [9]:
br.quit()
logging.info('хром драйвер выключен')

In [10]:
h = {'User-Agent': 'Mozilla/5.0'}
r = []
cache = {}
for i, it in enumerate(tqdm(d), 1):
    if len(r) >= target:
        break
    ln = it['link']
    if ln not in cache:
        logging.info(f'отправлен запрос к вакансиям на странице {ln}')
        try:
            resp = requests.get(ln, headers=h, timeout=30)
            resp.raise_for_status()
        except Exception as err:
            logging.error(f'Error for vacancy {ln} - {err}')
            time.sleep(slp2)
            continue
        sp = BeautifulSoup(resp.text, 'html.parser')
        t = sp.find('h1')
        ph = t.get_text(' ', strip=True) if t else ''
        cb = sp.select_one('.vacancy-company__title .link-comp--appearance-dark')
        fh = cb.get_text(' ', strip=True) if cb else ''
        db = sp.select_one('.vacancy-description__text') or sp.select_one('.content-wrapper__main')
        vc = db.get_text(' ', strip=True) if db else ''
        vc = re.sub(r'\s+', ' ', vc).strip()
        cache[ln] = (ph, fh, vc)
        time.sleep(slp2)
    ph, fh, vc = cache[ln]
    prof = ph if ph else it['profession']
    firm = fh if fh else it['firm_name']
    sl = str(it['salary_text']).strip()
    sl_low = sl.lower()
    pay_from = 0
    pay_to = 0
    if 'не указана' not in sl_low and sl != '':
        nums = re.findall(r'\d+', sl.replace(' ', '').replace('\xa0', ''))
        nums = [int(x) for x in nums]
        if len(nums) >= 2:
            pay_from = min(nums[0], nums[1])
            pay_to = max(nums[0], nums[1])
        elif len(nums) == 1:
            if 'до' in sl_low and 'от' not in sl_low:
                pay_to = nums[0]
            else:
                pay_from = nums[0]
    chips = it.get('meta_tokens', [])
    chips_low = ' '.join(chips).lower()
    if 'intern' in chips_low or 'junior' in chips_low or 'стаж' in chips_low:
        exp = 'Без опыта'
    elif 'senior' in chips_low:
        exp = 'От 3 лет'
    elif 'lead' in chips_low or 'head' in chips_low:
        exp = 'От 6 лет'
    else:
        exp = 'От 1 года'
    if 'удал' in chips_low or 'remote' in chips_low:
        place = 'Удалённая работа (на дому)'
    else:
        place = 'Не имеет значения'
    town = ''
    for x in chips:
        xl = x.lower()
        if any(k in xl for k in ['intern', 'junior', 'middle', 'senior', 'lead', 'head', 'удал', 'remote']):
            continue
        town = x
        break
    vcl = vc.lower()
    if 'вахт' in vcl:
        typ = 'Работа вахтовым методом'
    elif 'сменн' in vcl:
        typ = 'Сменный график работы'
    elif 'неполная дистанционная занятость' in vcl:
        typ = 'Неполная дистанционная занятость'
    elif 'частич' in vcl or 'неполная занятость' in vcl or 'совместительств' in vcl:
        typ = 'Частичная занятость / Совместительство'
    else:
        typ = 'Полный рабочий день'
    r.append({
        'vacancy_category': it['vacancy_category'],
        'profession': prof,
        'payment_from': pay_from,
        'payment_to': pay_to,
        'town': town,
        'experience': exp,
        'type_of_work': typ,
        'place_of_work': place,
        'firm_name': firm,
        'vacancy_clean': vc,
    })
    logging.info(f'успех для вакансии {i}')
logging.info(f'строк спарсено {len(r)}')

  0%|          | 0/10368 [00:00<?, ?it/s]

In [11]:
df = pd.DataFrame(r)
order_cols = ['vacancy_category', 'profession', 'payment_from', 'payment_to',
              'town', 'experience', 'type_of_work', 'place_of_work', 'firm_name', 'vacancy_clean']
df = df[order_cols]
df = df.drop_duplicates(subset=['vacancy_category', 'profession', 'firm_name', 'vacancy_clean'])
df = df.head(target).reset_index(drop=True)
df.to_csv('habr_df.csv', index=False, encoding='utf-8-sig')
logging.info(f'размер финального датафрейма {df.shape}')
logging.info('ура, парсинг хабра окончен')
df.head()

,vacancy_category,profession,payment_from,payment_to,town,experience,type_of_work,place_of_work,firm_name,vacancy_clean
0,AI,AI-Product,0,0,Москва,От 1 года,Полный рабочий день,Не имеет значения,PARI,Чем предстоит заниматься Запуск и развитие Saa...
1,AI,AI Automation/AI Systems Integrator (AI Automa...,0,0,Москва,От 1 года,Полный рабочий день,Не имеет значения,Торговый дом НЕБО,"Мы ищем гибридного инженера, который умеет соб..."
2,AI,"AI Backend Engineer (LLM, NLP, Voice AI)",0,400000,,От 1 года,Полный рабочий день,Удалённая работа (на дому),НТК,Мы создаём AI-платформу для анализа переговоро...
3,AI,Менеджер AI проектов,0,0,Москва,От 1 года,Полный рабочий день,Не имеет значения,АстраЗенека,О компании и команде AstraZeneca — это междуна...
4,AI,Prompt Engineer (AI / LLM),100000,0,Санкт-Петербург,От 3 лет,Полный рабочий день,Не имеет значения,infotech,О проекте Мы разрабатываем внутренний AI-проду...
